# LoRA/PEFT Fine-tuning — Llama-3.2-1B-Instruct

**Strategy:** PEFT (Parameter-Efficient Fine-Tuning) với LoRA (Low-Rank Adaptation)
— chỉ huấn luyện một số lượng nhỏ tham số adapter chèn vào các attention layers,
giảm yêu cầu GPU và bộ nhớ.

**Model:** `unsloth/Llama-3.2-1B-Instruct` (1B params)

**Dataset:** `code_x_glue_cc_defect_detection`

**Ưu điểm:** Tiết kiệm bộ nhớ, train nhanh, hiệu quả gần bằng full fine-tuning.

**Nhược điểm:** Cần tuning hyperparameters (rank, alpha, target modules).

## 1. Setup

In [ ]:
import os

# --- Colab auto-setup ---
IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")
REPO_URL = "https://github.com/MinhTuan2405/IE105.21_SBugDwLLM.git"
BRANCH = "trung"
NOTEBOOK_SUBDIR = "finetuning/llama32"

if IN_COLAB:
    REPO_DIR = f"/content/IE105.21_SBugDwLLM"
    if not os.path.exists(REPO_DIR):
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    os.chdir(os.path.join(REPO_DIR, NOTEBOOK_SUBDIR))
    print(f"Working directory: {os.getcwd()}")

!pip install -q unsloth
!pip install -q transformers datasets accelerate peft trl bitsandbytes
!pip install -q scikit-learn matplotlib seaborn huggingface_hub

from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass

if hf_token:
    login(token=hf_token)
    print("Logged in with HF_TOKEN")
else:
    login()  # Fallback: interactive prompt

In [ ]:
import sys
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from trl import SFTTrainer
import trl
print(f"trl version: {trl.__version__}")

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from utils.data_loader import load_defect_dataset, format_for_finetuning, build_chat_messages, SYSTEM_PROMPT
from utils.evaluation import evaluate_predictions, parse_model_output

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
MODEL_ID = "unsloth/Llama-3.2-1B-Instruct"
OUTPUT_DIR = "./llama32-lora-finetuned"
RESULTS_DIR = "../../docs/llama32_docs"

MAX_SEQ_LENGTH = 512

# LoRA hyperparameters
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# Training hyperparameters
NUM_EPOCHS = 3
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 2e-4
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01

## 3. Load Dataset

In [ ]:
train_data, val_data, test_data = load_defect_dataset()
print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

train_labels = [s["target"] for s in train_data]
print(f"Distribution: 0={train_labels.count(0)}, 1={train_labels.count(1)}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

train_formatted = train_data.map(lambda x: format_for_finetuning(x, tokenizer=tokenizer))
val_formatted = val_data.map(lambda x: format_for_finetuning(x, tokenizer=tokenizer))

print("\nSample:")
print(train_formatted[0]["text"][:500])

## 4. Load Model with 4-bit Quantization

In [ ]:
model, _ = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
print(f"Base model loaded. Memory: {model.get_memory_footprint() / 1e9:.2f} GB")

## 5. Apply LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
model.print_trainable_parameters()

## 6. Training

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    optim="adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    dataset_text_field="text",
    packing=True,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Starting LoRA fine-tuning...")
trainer.train()

In [ ]:
trainer.save_model(os.path.join(OUTPUT_DIR, "best"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "best"))
print(f"LoRA adapters saved to {OUTPUT_DIR}/best")

## 7. Evaluation

In [ ]:
from tqdm import tqdm

FastLanguageModel.for_inference(model)
device = next(model.parameters()).device
y_true, y_pred, failed_parses = [], [], []

for i, sample in enumerate(tqdm(test_data, desc="Evaluating")):
    messages = build_chat_messages(sample["func"])
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=5, do_sample=False)

    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    pred = parse_model_output(generated)

    if pred == -1:
        failed_parses.append({"index": i, "output": generated, "label": int(sample["target"])})
        pred = 0

    y_true.append(int(sample["target"]))
    y_pred.append(pred)

print(f"\nFailed to parse: {len(failed_parses)} / {len(test_data)}")

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

metrics = evaluate_predictions(
    y_true, y_pred,
    model_name="llama32",
    strategy="lora_finetuning",
    save_dir=RESULTS_DIR,
)

## 8. Error Analysis

In [ ]:
errors = [
    {"index": i, "true": yt, "pred": yp, "code": test_data[i]["func"][:200]}
    for i, (yt, yp) in enumerate(zip(y_true, y_pred))
    if yt != yp
]

false_positives = [e for e in errors if e["true"] == 0 and e["pred"] == 1]
false_negatives = [e for e in errors if e["true"] == 1 and e["pred"] == 0]

print(f"Total errors: {len(errors)}")
print(f"False Positives: {len(false_positives)}")
print(f"False Negatives: {len(false_negatives)}")

print("\n--- Sample False Negatives ---")
for e in false_negatives[:3]:
    print(f"  idx={e['index']}: {e['code'][:100]}...")

print("\n--- Sample False Positives ---")
for e in false_positives[:3]:
    print(f"  idx={e['index']}: {e['code'][:100]}...")

## 9. So sánh LoRA vs Full Fine-tuning

Sau khi chạy xong cả hai notebook, so sánh kết quả tại `docs/llama32_docs/`.

In [ ]:
import json
import glob

result_files = glob.glob(os.path.join(RESULTS_DIR, "llama32_*.json"))
if result_files:
    print("\n" + "="*60)
    print("  ALL RESULTS SO FAR")
    print("="*60)
    for f in sorted(result_files):
        with open(f) as fp:
            r = json.load(fp)
        print(f"\n{r['strategy']}:")
        for k, v in r['metrics'].items():
            print(f"  {k:>12s}: {v:.4f}")